## Task 2: Uncertainty estimation before and after calibration

#### Goal: Assess the effect of probability calibration on prediction uncertainty.

1. Evaluate ensemble accuracy before and after calibration on the test set. Compare the two using McNemar's test.
2. Compute and plot the distribution of **total uncertainty** (predictive entropy) before and after calibration:
$$H = -\sum_c \bar{p}_c \log \bar{p}_c$$
where $\bar{p}_c$ is the mean predicted probability for class $c$ across ensemble members.
3. Compute and plot the distribution of **epistemic uncertainty** (mutual information between predictions and model parameters) before and after calibration:
$$U_{\text{epistemic}} = H(\bar{p}) - \frac{1}{K}\sum_{k=1}^K H(p^{(k)})$$
4. Compute and plot the distribution of **aleatoric uncertainty** (expected entropy of individual model predictions) before and after calibration:
$$U_{\text{aleatoric}} = \frac{1}{K}\sum_{k=1}^K H(p^{(k)})$$

**Assignment:** Compare classifier accuracy and uncertainty distributions before and after probability calibration.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from utils import *

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=5000, n_features=20)
X, y = torch.tensor(X, dtype=torch.float), torch.tensor(y)

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2)

model = MLP(input_size=20, hidden_layers=[64, 256, 64], output_size=2)

fold_models = []
fold_models = fold_train(X_trainval=X_trainval, y_trainval=y_trainval,
                         model=model, EPOCHS=2, device=device)

ensemble = [load_model(k, model, device) for k in fold_models]

scalers = []
for i, m in enumerate(ensemble):
    print(f"Fitting scaler for model {i+1}/5...")
    s = fit_temperature(m, X_test, y_test)
    scalers.append(s)

Fold 1/5 — best val loss: 0.6030
Fold 2/5 — best val loss: 0.5971
Fold 3/5 — best val loss: 0.5912
Fold 4/5 — best val loss: 0.5971
Fold 5/5 — best val loss: 0.5941
Fitting scaler for model 1/5...
  Fitted T = 2.2473
Fitting scaler for model 2/5...
  Fitted T = 2.2303
Fitting scaler for model 3/5...
  Fitted T = 2.2334
Fitting scaler for model 4/5...
  Fitted T = 2.2354
Fitting scaler for model 5/5...
  Fitted T = 2.2398


In [7]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

y_true = y_test.numpy()
X_test_dev = X_test.to(device)

probs_before = get_probs_ensemble(ensemble, X_test, device)
probs_after = get_probs_ensemble(scalers, X_test, device)

preds_before = probs_before.argmax(axis=1)
preds_after = probs_after.argmax(axis=1)

correct_before = (preds_before == y_true)
correct_after = (preds_after == y_true)

table = np.array([
    [(~correct_before & ~correct_after).sum(), (correct_before & ~correct_after).sum()],
    [(~correct_before & correct_after).sum(), (correct_before & correct_after).sum()]
])

result = mcnemar(table, exact=False, correction=True)

acc_before = correct_before.mean()
acc_after = correct_after.mean()

print(f"Accuracy before: {acc_before:.4f}")
print(f"Accuracy after: {acc_after:.4f}")
print(f"McNemar statistic: {result.statistic:.4f},  p-value: {result.pvalue:.4f}")
print(f"→ {'Brak istotnej różnicy w accuracy' if result.pvalue > 0.05 else 'Istotna różnica w accuracy (p < 0.05)'}")

Accuracy before: 0.8280
Accuracy after: 0.8280
McNemar statistic: inf,  p-value: 0.0000
→ Istotna różnica w accuracy (p < 0.05)
